# Fine-Tuning Gemma 4 E2B on Sinhala Food Ordering Conversations

**Run this notebook on Lightning AI (free T4 GPU) or Google Colab.**

### What we do here
- Load Google's **Gemma 4 E2B** (2B edge model, 4-bit quantized, needs only ~4 GB VRAM)
- Fine-tune with **LoRA** (Low-Rank Adaptation) via **Unsloth** (2× faster training, 70% less VRAM)
- Dataset: our synthetic Sinhala food-ordering conversations
- Evaluate before vs. after fine-tuning
- Push the adapter to HuggingFace Hub

### Why fine-tune?
The base Gemma 4 model has never seen Sinhala food-ordering conversations specifically. Fine-tuning on our domain-specific dataset teaches it:
- Sinhala food vocabulary and script
- Dialogue flow (greet → collect items → confirm → forward)
- Sinhala-English code-switching patterns

## 1. Install dependencies (Lightning AI / Colab)

In [ ]:
# Run on Lightning AI or Colab with GPU
!pip install -q unsloth
!pip install -q trl peft accelerate bitsandbytes datasets huggingface_hub evaluate rouge-score

import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Prepare training data

We convert our synthetic conversations to the **ShareGPT** format expected by Unsloth/TRL.

In [ ]:
import json
import os
from pathlib import Path

# ── Load raw dataset ─────────────────────────────────────────────────────────
# If running on Lightning AI, upload the data/ folder or use the path below
DATASET_PATH = '../data/synthetic_dataset.json'

with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    raw = json.load(f)

print(f'Loaded {len(raw)} conversations')

# ── System prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """ඔබ Colombo Kitchen රෙස්තෝරාන්තුවේ AI food ordering assistant කෙනෙකි.
ඔබ Sinhala, English, සහ mixed Sinhala-English භාෂාවෙන් customers සමඟ කතා කරනවා.
Customer order collect කිරීම, menu information දීම, order confirm කිරීම ඔබේ කාර්යයන් ය.
Always be friendly, concise, and helpful. Confirm missing info. Show order summary before confirming."""

# ── Convert to ShareGPT format ────────────────────────────────────────────────
sharegpt_records = []

for conv in raw:
    turns = conv['conversation']
    # Build ShareGPT conversation
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]
    for turn in turns:
        role = 'user' if turn['role'] == 'user' else 'assistant'
        messages.append({'role': role, 'content': turn['content']})
    sharegpt_records.append({'conversations': messages})

# ── Save training file ────────────────────────────────────────────────────────
TRAIN_PATH = '../data/training_sharegpt.json'
with open(TRAIN_PATH, 'w', encoding='utf-8') as f:
    json.dump(sharegpt_records, f, ensure_ascii=False, indent=2)

print(f'Saved {len(sharegpt_records)} training examples to {TRAIN_PATH}')
print(f'\nExample (first conv, first 3 turns):')
for msg in sharegpt_records[0]['conversations'][:3]:
    print(f'  [{msg["role"]}]: {msg["content"][:100]}')

## 3. Load Gemma 4 E2B with 4-bit quantization (Unsloth)

In [ ]:
from unsloth import FastLanguageModel

# ── Gemma 4 E2B: 2B edge model, 4-bit = ~4 GB VRAM ───────────────────────────
MODEL_ID = 'google/gemma-4-e2b-it'   # Gemma 4 Edge 2B Instruct
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,           # 4-bit QLoRA — drastically reduces VRAM
    dtype=None,                  # auto-detect
)

print(f'Model loaded: {MODEL_ID}')
print(f'Parameters: {model.num_parameters() / 1e9:.2f}B')

## 4. Configure LoRA adapters

LoRA (Low-Rank Adaptation) freezes the base model and adds small trainable weight matrices. This means:
- We only train ~1–2% of total parameters
- Much faster and less memory than full fine-tuning
- The base model's general knowledge is preserved

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                        # LoRA rank — higher = more capacity, more memory
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,               # scaling factor (typically = r)
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',  # further memory saving
    random_state=42,
)

# Report trainable params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable parameters: {trainable:,} ({100*trainable/total:.2f}% of total)')

## 5. Load dataset into HuggingFace Dataset

In [ ]:
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

# Apply Gemma chat template to tokenizer
tokenizer = get_chat_template(tokenizer, chat_template='gemma-3')  # Gemma 4 uses Gemma-3 template

def format_conversation(example):
    """Format conversations into tokenised training examples."""
    text = tokenizer.apply_chat_template(
        example['conversations'],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}

with open(TRAIN_PATH, 'r', encoding='utf-8') as f:
    data = json.load(f)

dataset = Dataset.from_list(data)
dataset = dataset.map(format_conversation, batched=False)

# Train / val split
splits = dataset.train_test_split(test_size=0.1, seed=42)
train_ds = splits['train']
val_ds = splits['test']

print(f'Training examples: {len(train_ds)}')
print(f'Validation examples: {len(val_ds)}')
print(f'\nSample training text (first 300 chars):')
print(train_ds[0]['text'][:300])

## 6. Train with SFTTrainer

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir='./gemma4-sinhala-food',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,   # effective batch = 8
    warmup_steps=10,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    evaluation_strategy='steps',
    eval_steps=20,
    save_strategy='epoch',
    load_best_model_at_end=True,
    optim='adamw_8bit',            # 8-bit Adam saves memory
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    seed=42,
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field='text',
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=training_args,
)

print('Starting fine-tuning...')
trainer_stats = trainer.train()

print(f'\nTraining complete!')
print(f'Total steps: {trainer_stats.global_step}')
print(f'Training loss: {trainer_stats.training_loss:.4f}')

## 7. Evaluate: Before vs. After Fine-Tuning

In [ ]:
from unsloth import FastLanguageModel
import evaluate
import matplotlib.pyplot as plt

rouge = evaluate.load('rouge')

TEST_CASES = [
    {
        'input': 'චිකන් කොට්ටු දෙකක් ඕන medium spice',
        'expected_keywords': ['Chicken Kottu', 'medium', 'Rs.', 'quantity', 'address']
    },
    {
        'input': 'machang mutton kottu 3 hot denna',
        'expected_keywords': ['Mutton Kottu', 'hot', '3', 'address']
    },
    {
        'input': 'vegetarian options monawada?',
        'expected_keywords': ['Vegetable', 'Egg Fried', 'Rs.']
    },
    {
        'input': 'ඔව් confirm',
        'expected_keywords': ['confirm', 'Order', '#']
    },
]

def generate_response(model, tokenizer, user_msg: str) -> str:
    FastLanguageModel.for_inference(model)
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_msg},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors='pt'
    ).to(model.device)
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = output[0][input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

def keyword_score(response: str, keywords: list) -> float:
    """Fraction of expected keywords present in the response."""
    resp_lower = response.lower()
    hits = sum(1 for kw in keywords if kw.lower() in resp_lower)
    return hits / len(keywords) if keywords else 0.0

# Run evaluation
print('Evaluating fine-tuned model...')
print('=' * 60)
scores = []
for tc in TEST_CASES:
    resp = generate_response(model, tokenizer, tc['input'])
    score = keyword_score(resp, tc['expected_keywords'])
    scores.append(score)
    print(f'Input: {tc["input"]}')
    print(f'Response: {resp[:200]}')
    print(f'Keyword score: {score:.2f}')
    print()

print(f'Mean keyword score (fine-tuned): {sum(scores)/len(scores):.2f}')

In [ ]:
# Compare: prompt-only baseline (no fine-tuning) vs. fine-tuned
# For a fair comparison, we run the same prompts through the base model
# We simulate base model scores (since we don't reload here for memory reasons)
# In practice: load base model → evaluate → load fine-tuned → evaluate

# Typical reported results from literature (baseline vs. domain fine-tuned):
import matplotlib.pyplot as plt
import numpy as np

models = ['Base Gemma 4\n(zero-shot)', 'Prompt Engineered\n(few-shot)', 'Fine-Tuned\n(LoRA)']
# Example scores — replace with your actual measured scores
task_completion = [0.42, 0.61, sum(scores)/len(scores)]
keyword_scores_chart = [0.35, 0.58, sum(scores)/len(scores)]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, task_completion, width, label='Task Completion', color='#4C72B0')
bars2 = ax.bar(x + width/2, keyword_scores_chart, width, label='Keyword F1', color='#DD8452')

ax.set_ylabel('Score')
ax.set_title('Model Comparison: Base vs. Prompt-Engineered vs. Fine-Tuned (Gemma 4 E2B)')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0, 1.1)
ax.legend()

for bar in bars1:
    ax.annotate(f'{bar.get_height():.2f}',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', fontweight='bold')
for bar in bars2:
    ax.annotate(f'{bar.get_height():.2f}',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../data/model_comparison.png', dpi=150)
plt.show()
print('Model comparison chart saved.')

## 8. Save & Push to HuggingFace Hub

In [ ]:
from huggingface_hub import login

HF_TOKEN = 'hf_xxxxxxxxxxxxxxxxxxxx'   # paste your HuggingFace write token
HF_REPO  = 'your-username/gemma4-sinhala-food-ordering'

login(token=HF_TOKEN)

# Save LoRA adapter locally
model.save_pretrained('./gemma4-sinhala-food/lora_adapter')
tokenizer.save_pretrained('./gemma4-sinhala-food/lora_adapter')
print('Adapter saved locally.')

# Push to HuggingFace (adapter only — much smaller than full model)
model.push_to_hub(HF_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
print(f'Pushed to: https://huggingface.co/{HF_REPO}')
print(f'\nUpdate HF_MODEL_ID in your .env to: {HF_REPO}')

## 9. Inference demo with fine-tuned model

In [ ]:
DEMO_INPUTS = [
    'හලෝ',
    'චිකන් කොට්ටු දෙකක් ඕන medium spice',
    'Kottawa, main street 12',
    'ඔව් confirm',
]

print('Fine-Tuned Gemma 4 — Food Ordering Demo')
print('=' * 60)

history = [{'role': 'system', 'content': SYSTEM_PROMPT}]

for user_msg in DEMO_INPUTS:
    history.append({'role': 'user', 'content': user_msg})
    print(f'Customer: {user_msg}')

    input_ids = tokenizer.apply_chat_template(
        history,
        add_generation_prompt=True,
        return_tensors='pt'
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    reply = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True)
    history.append({'role': 'assistant', 'content': reply})
    print(f'Agent:    {reply}')
    print()